# Train Model (for hls4ml)

## Libraries

In [ ]:
import os

In [ ]:
# Remove TF warnings (this can be dangerous)
os.environ["TF_XLA_FLAGS"] = "--tf_xla_enable_xla_devices"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

In [ ]:
# See: https://gist.github.com/zrruziev/b93e1292bf2ee39284f834ec7397ee9f
# sudo echo 0 | sudo tee -a /sys/bus/pci/devices/0000\:01\:00.0/numa_node

In [ ]:
import tensorflow as tf
import tensorflow.keras
from keras import datasets, layers, models
from keras.optimizers import Adam
from keras.models import load_model
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *
from qkeras.utils import _add_supported_quantized_objects

from keras.utils import Sequence
from keras.callbacks import CSVLogger
from keras.callbacks import EarlyStopping

import random
from datetime import datetime
import time

pi = 3.14159265359

maxval=1e9
minval=1e-9

## GPUs

In [ ]:
# You can disable the GPU, if a GPU is present
#os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

## Import Model and Loss functions

In [ ]:
from dataloaders.OptimizedDataGenerator_v2p5 import OptimizedDataGenerator
from loss import *
from mlp_encoder_model import *

## Configuration

In [ ]:
seed = 10
tf.random.set_seed(seed)
random.seed(seed)

In [ ]:
load_model_from_file_enabled = True
load_tfr_from_file_enabled = True
randomize_dataset_enabled = False

### Directories

In [ ]:
#dataset_base_dir = "/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/"
dataset_base_dir = "/nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/"

tfrecords_base_dir = os.path.join(dataset_base_dir, "TFR_files", "2t")
npy_base_dir = os.path.join(dataset_base_dir, "NPY_files", "2t")

dataset_train_dir = os.path.join(dataset_base_dir, "train_contained")
dataset_validation_dir = os.path.join(dataset_base_dir, "test_contained")

npy_dir_val = os.path.join(npy_base_dir, "NPY_val_contained_slim")
tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train_contained_slim")
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val_contained_slim")

dirs_to_create = [
    tfrecords_dir_train,
    tfrecords_dir_val,
    dataset_train_dir,
    dataset_validation_dir,
    npy_dir_val
]

# Create each directory if it doesn't exist
for directory in dirs_to_create:
    os.makedirs(directory, exist_ok=True)

In [ ]:
!ls $dataset_base_dir
!ls $dataset_base_dir/NPY_files/2t/NPY_val_contained_slim

In [ ]:
print(f'Number of training files: {len(os.listdir(dataset_train_dir))}')
print(f'Number of validation files: {len(os.listdir(dataset_validation_dir))}')

### Hyper-parameters

In [ ]:
batch_size = 5000
val_batch_size = 5000
train_file_size = len(os.listdir(dataset_train_dir))
val_file_size = len(os.listdir(dataset_validation_dir))

## Data Generation

In [ ]:
start_time = time.time()
validation_generator = OptimizedDataGenerator(
    dataset_base_dir = dataset_validation_dir,
    file_type = "parquet",
    data_format = "3D",
    batch_size = val_batch_size,
    file_count = val_file_size,
    to_standardize = False, # False when processing manually digitized inputs
    log_compression = False, # False when processing manually digitized inputs
    select_contained = True,
    noise = -1,
    min_threshold = None,
    max_threshod = None,
    include_y_local= False,
    labels_list = ['x-midplane','y-midplane','cotBeta'],
    input_shape = (2,16,16), # (20,16,16),
    transpose = (0,2,3,1),
    shuffle = False, 
    files_from_end = True,

    load_from_tfrecords_dir = tfrecords_dir_val if load_tfr_from_file_enabled else None,
    tfrecords_dir = tfrecords_dir_val,
    use_time_stamps = [0,19],
    max_workers = 2,
    #load_from_tfrecords_dir = tfrecords_dir_val
)

print("--- Validation generator %s seconds ---" % (time.time() - start_time))

In [ ]:
# training generator
start_time = time.time()
training_generator = OptimizedDataGenerator(
    dataset_base_dir = dataset_train_dir,
    file_type = "parquet",
    data_format = "3D",
    batch_size = batch_size,
    file_count = train_file_size,
    to_standardize = False, # False when processing manually digitized inputs
    log_compression = False, # False when processing manually digitized inputs
    select_contained = True,
    noise = -1,
    min_threshold = None,
    max_threshold = None,
    include_y_local= False,
    labels_list = ['x-midplane','y-midplane','cotBeta'],
    input_shape = (2,16,16), # (20,16,16),
    transpose = (0,2,3,1),
    shuffle = False, # True 

    load_from_tfrecords_dir = tfrecords_dir_train if load_tfr_from_file_enabled else None,
    tfrecords_dir = tfrecords_dir_train,
    use_time_stamps = [0,19],
    max_workers = 2,
    #load_from_tfrecords_dir = tfrecords_dir_train
)
print("--- Training generator %s seconds ---" % (time.time() - start_time))

In [ ]:
if randomize_dataset_enabled:
    training_generator = OptimizedDataGenerator(
        load_from_tfrecords_dir = tfrecords_dir_train,
        shuffle = True,
        seed = seed,
        quantize = False,
    )

    validation_generator = OptimizedDataGenerator(
        load_from_tfrecords_dir = tfrecords_dir_val,
        shuffle = True,
        seed = seed,
        quantize = False,
    )
else:
    print("Disable dataset randomization")

In [ ]:
# Create a numpy array that contains the recon3D and labels information
# X is the 2-timeslices recon3D data
# y is the labels data ['x-midplane','y-midplane','cotAlpha','cotBeta']
X_val_all = []
y_val_all = []

num_batches = validation_generator.__len__() # The total number of batches for the validation dataset
#val_num_batches = 1

for i_batch in range(num_batches): # Loop over all batches
    X_val, y_val = validation_generator.__getitem__(i_batch)
    X_val = X_val.numpy()
    y_val = y_val.numpy()
    X_val_all.append(X_val)
    y_val_all.append(y_val)

X_val_all = np.array(np.concatenate(X_val_all))
y_val_all = np.array(np.concatenate(y_val_all))

np.save(f"{npy_dir_val}/X_val.npy", X_val_all)
np.save(f"{npy_dir_val}/y_val.npy", y_val_all)

## Create Model

In [ ]:
model=CreateModel_Slim((16,16,2))
model.compile(
    optimizer=tf.keras.optimizers.Nadam(learning_rate=1e-3),
    loss=custom_sse_loss
)

model.summary()

### Weights Directories

In [ ]:
# training
#base_dir = '/data/dajiang/smart-pixels/weights/dataset_3src_16x16_weights/'
base_dir = 'weights/dataset_3src_16x16_weights/'
!ls $base_dir

pitch = '50x12P5'
fingerprint = '%08x' % random.randrange(16**8)
print(fingerprint)

In [ ]:
!ls $base_dir
!ls $base_dir/weights-50x12P5-bs5000-34c2da80-2t-mlp_SLIM-model_quantized-checkpoints/*hdf5 | wc -l
!ls $base_dir/weights-50x12P5-bs5000-34c2da80-2t-mlp_SLIM-model_quantized-checkpoints/*json

In [ ]:
weights_dir = base_dir + 'weights-{}-bs{}-{}-2t-mlp_SLIM-model_quantized-checkpoints'.format(pitch, batch_size, fingerprint)
best_model_hdf5 = f"{weights_dir}/best_model-{pitch}-bs{batch_size}-{fingerprint}-2t-mlp_SLIM-model_quantized.hdf5"
best_model_keras = f"{weights_dir}/best_model-{pitch}-bs{batch_size}-{fingerprint}-2t-mlp_SLIM-model_quantized.keras"
best_model_weights_hdf5 = f"{weights_dir}/best_model_weights-{pitch}-bs{batch_size}-{fingerprint}-2t-mlp_SLIM-model_quantized.hdf5"
best_model_weights_keras = f"{weights_dir}/best_model_weights-{pitch}-bs{batch_size}-{fingerprint}-2t-mlp_SLIM-model_quantized.keras"
model_architecture_json = f"{weights_dir}/model_architecture-{pitch}-bs{batch_size}-{fingerprint}-2t-mlp_SLIM-model_quantized.json"

if not load_model_from_file_enabled:
    # create output directories
    !rm -rf $weights_dir
    if os.path.isdir(base_dir):
        os.mkdir(weights_dir)
    else:
        os.mkdir(base_dir)
        os.mkdir(weights_dir)
    
    # es = EarlyStopping(
    #     patience=50,
    #     restore_best_weights=True
    # )
    
    checkpoint_filepath = weights_dir + '/weights.{epoch:02d}-t{loss:.2f}-v{val_loss:.2f}.hdf5'
    mcp = tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_filepath,
        save_weights_only=True,
        monitor='val_loss',
        save_best_only=False,
    )

    print('Model fingerprint: {}'.format(fingerprint))

else:
    pass

## Training

In [ ]:
if not load_model_from_file_enabled:
    history = model.fit(x=training_generator,
                    validation_data=validation_generator,
                    callbacks=[mcp],
                    epochs=2000,
                    shuffle=False, # shuffling now occurs within the data-loader
                    verbose=1)
    
    # Revert to best model
    files = os.listdir(weights_dir)
    vlosses = [float(f.split("-v")[1].split(".hdf5")[0]) for f in files]
    bestfile = files[np.argmin(vlosses)]
    model.load_weights(f"{weights_dir}/{bestfile}")

    # Save (best) model information to file
    model.save(best_model_hdf5)
    model.save(best_model_keras)
    model.save_weights(best_model_weights_hdf5)
    model.save_weights(best_model_weights_keras)
    model_json = model.to_json()
    with open(model_architecture_json, "w") as json_file:
        json_file.write(model_json)
else:
    co = {"custom_sse_loss": custom_sse_loss}
    _add_supported_quantized_objects(co)
    # This overrides the previously compiled model
    # TODO: load just weights
    model = load_model(best_model_hdf5, custom_objects=co)
    model.summary()

In [ ]:
training_validation_loss_png = f"training_validation_loss.png"
if load_model_from_file_enabled:
    from PIL import Image
    import matplotlib.pyplot as plt
    img = Image.open(training_validation_loss_png)
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else: 
    import matplotlib.pyplot as plt
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(training_validation_loss_png, bbox_inches='tight', pad_inches=0.5)
    plt.show()